#  Procesamiento Digital de Imágenes - TP06: Restauración de imágenes

**Objetivos de la guía:**

* Comprender la dinámica de los modelos de ruido y sus parámetros.

* Formular estrategias de filtrado en dominio espacial y frecuencial

* Experimentar con filtros de medias, de orden y sus efectos en cascada.

* Estudiar los efectos de los filtros de deconvolución.

In [ ]:
# Configuración Inicial y Función de Error
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import ipywidgets as widgets
import scipy.ndimage as ndimage # Útil para filtros no lineales

def mostrar_imagenes(imagenes, titulos, figsize=(15, 5)):
    n = len(imagenes)
    fig, axs = plt.subplots(1, n, figsize=figsize)
    if n == 1: axs = [axs]
    for i in range(n):
        img = imagenes[i]
        if len(img.shape) == 3: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axs[i].imshow(img, cmap='gray', vmin=0, vmax=255)
        axs[i].set_title(titulos[i])
        axs[i].axis('off')
    plt.tight_layout()
    plt.show()

# Cómputo del ECM (MSE, mean square error) provisto en la guía
def mse(imageA, imageB):
    err = np.sum((imageA.astype("float") - imageB.astype("float")) ** 2)
    err /= float(imageA.shape * imageA.shape[1])
    return err

print("Entorno configurado. ¡Asegúrate de subir las imágenes necesarias ('sangre.jpg', 'img_degradada.tif', imágenes FAMILIA, etc.)!")

## Ejercicio 1: Modelos de ruido

Generar distintos tipos de ruido:
* ruido gaussiano,
* ruido uniforme,
* sal y pimienta,
* impulsivo
* exponencial

Analizar su impacto sobre una imagen de prueba compuesta por 3 franjas de grises constantes.


In [ ]:
#  Generación de Ruido y Análisis de Histogramas

def modelos_de_ruido(media_gauss=0, std_gauss=20):
    # 1. Generar imagen de 600x600 con 3 franjas verticales de grises
    img_franjas = np.zeros((600, 600), dtype=np.uint8)
    img_franjas[:, :200] = 60   # Oscuro
    img_franjas[:, 200:400] = 120 # Medio
    img_franjas[:, 400:] = 180  # Claro

    # 2. Generar Ruido Gaussiano con media 0
    # TODO: Usa np.random.normal(mean, sigma, (row,col))
    # ruido_gaussiano = ...
    ruido_gaussiano = np.zeros((600,600), dtype=np.float32) # Reemplazar

    # 3. Sumar el ruido a la imagen (¡Cuidado con los desbordamientos!)
    img_ruidosa = np.clip(img_franjas.astype(np.float32) + ruido_gaussiano, 0, 255).astype(np.uint8)

    # Visualización
    fig, axs = plt.subplots(2, 2, figsize=(10, 8))
    axs.imshow(img_franjas, cmap='gray', vmin=0, vmax=255); axs.set_title("Original")
    axs[1].hist(img_franjas.ravel(), bins=256, range=); axs[1].set_title("Histograma Original")

    axs[1].imshow(img_ruidosa, cmap='gray', vmin=0, vmax=255); axs[1].set_title("Con Ruido Gaussiano")
    axs[1].hist(img_ruidosa.ravel(), bins=256, range=); axs[1].set_title("Histograma Ruidoso")
    plt.show()

    # TODO: Repite el proceso para ruido uniforme, sal y pimienta, impulsivo y exponencial.
    # Recuerda adaptar las funciones para que los ruidos tengan media cero.

interact(modelos_de_ruido, media_gauss=FloatSlider(min=-50, max=50, value=0), std_gauss=FloatSlider(min=1, max=100, value=20))

**Reflexiona sobre lo que observas:**

* Al observar los histogramas de la imagen con ruido gaussiano añadido, ¿qué relación notas entre la desviación estándar del ruido y el ancho de la "campana" que se forma en cada nivel de gris?

* ¿Qué le ocurriría al histograma en los bordes (0 y 255) si en lugar de ruido gaussiano aplicáramos ruido "sal y pimienta"?

## Ejercicio 2: Filtros de medias

Implementaremos filtros de media geométrica y contra-armónica sobre la imagen **sangre.jpg** contaminada con ruido mixto.

In [ ]:
# Filtros de Medias y ECM
def filtros_de_medias(img_path='sangre.jpg'):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return "Sube 'sangre.jpg'"

    # 1. Contaminar con mezcla de ruido impulsivo y gaussiano
    # TODO: Genera ruido gaussiano y ruido impulsivo (sal y pimienta) y súmalos a 'img'
    img_degradada = img.copy() # Reemplazar con imagen contaminada

    img_float = img_degradada.astype(np.float32)

    # 2. Filtro de Media Geométrica
    # Hint: Multiplicatoria de los pixeles en la vecindad elevada a la (1/mn).
    # Puede implementarse con exp(filtro_promedio(log(img))) para evitar desbordes matemáticos.
    # TODO: Implementar media geométrica
    img_media_geom = np.zeros_like(img) # Reemplazar

    # 3. Filtro de Media Contra-armónica
    # Hint: Numerador = suma(pixeles^(Q+1)) / Denominador = suma(pixeles^Q)
    # TODO: Implementar media contra-armónica con un parámetro Q elegido
    Q = 1.5
    img_contra_arm = np.zeros_like(img) # Reemplazar

    # 4. Evaluación cuantitativa (ECM/MSE)
    ecm_degradada = mse(img, img_degradada)
    ecm_geom = mse(img, img_media_geom)
    ecm_contra = mse(img, img_contra_arm)

    print(f"ECM Degradada vs Original: {ecm_degradada:.2f}")
    print(f"ECM Media Geométrica: {ecm_geom:.2f}")
    print(f"ECM Media Contra-armónica (Q={Q}): {ecm_contra:.2f}")

    mostrar_imagenes([img_degradada, img_media_geom, img_contra_arm],
                     ["Degradada", "Media Geométrica", f"Contra-armónica (Q={Q})"])

filtros_de_medias()


**Reflexiona sobre lo que observas:**

* Revise la fórmula matemática de la media contra-armónica.
  * ¿Qué efecto tiene utilizar un parámetro $Q$ positivo frente a ruido tipo "sal" (puntos blancos)?
  * ¿Y si utiliza un $Q$ negativo frente a ruido tipo "pimienta" (puntos negros)?

* ¿Le parece que un filtro basado en promedios es la mejor herramienta matemática para eliminar ruido impulsivo severo? ¿Si/No, Por qué?

##  Ejercicio 3: Filtros de orden

Utilizaremos filtros basados en ordenamiento estadístico (mediana, punto medio, media-alfa) para contrastar su desempeño contra los filtros de medias.

In [ ]:
#  Filtros Estadísticos y Cascada

def filtros_de_orden(img_degradada):
    # Se asume que recibe la imagen degradada del Ejercicio 2

    # a) Filtro de mediana
    # TODO: Usar cv2.medianBlur(src, ksize)
    img_mediana = np.zeros_like(img_degradada)

    # b) Filtro del punto medio
    # Hint: (Max(vecindad) + Min(vecindad)) / 2. Puedes usar ndimage.maximum_filter y minimum_filter
    # TODO: Implementar punto medio
    img_punto_medio = np.zeros_like(img_degradada)

    # c) Filtro de media-alfa recortado
    # TODO: Ordenar la vecindad, recortar 'd' elementos de los extremos, y promediar el resto.
    img_alfa_recortado = np.zeros_like(img_degradada)

    # d) Aplicación en cascada: filtro (a) seguido de (b)
    # TODO: Aplicar punto_medio(mediana(img))
    img_cascada = np.zeros_like(img_degradada)

    # TODO: Calcular ECM para cada salida comparando contra la imagen limpia original

    # Visualizar histogramas antes y después (implementar)

# Recuerde conectar con el Ejercicio 2 y correr.


**Reflexiona sobre lo que observas:**

* Considere el funcionamiento interno del filtro de mediana. Si tiene un pixel de ruido "sal" (intensidad 255) en una vecindad de 3x3, ¿bajo qué condiciones matemáticas específicas ese pixel sobreviviría y seguiría mostrándose tras aplicar el filtro?


* Al aplicar los filtros en cascada en el inciso (d), ¿qué tipo de ruido intenta remover primero y por qué el segundo filtro resulta un buen complemento?

## Ejercicio 4: Filtro adaptativo de reducción local del ruido

Filtro que ajusta su comportamiento basándose en la varianza estadística local de la imagen.

In [ ]:
# Filtro Adaptativo

def filtro_adaptativo():
    # 1. Cargar imagen y agregar ruido gaussiano (media 0, var 0.01)
    # Hint: varianza 0.01 en rango 0-1 equivale a un desvío estándar específico en rango 0-255.

    # 2. Implementar algoritmo
    # Fórmula: f_hat = g - (var_ruido / var_local) * (g - media_local)
    # TODO: Calcular media_local y var_local usando cv2.blur o ndimage para cada pixel

    # 3. Comparar con Media Geométrica del Ejercicio 2

    # 4. Repetir para varianzas mayores y evaluar resultados.
    pass


**Reflexiona sobre lo que observas:**

* Observe la fórmula del filtro adaptativo. En zonas planas de la imagen, la varianza local ($σ^{2}_{L}$) será casi igual a la varianza global del ruido ($σ^{2}_{\eta}$).
¿En qué se convierte la ecuación matemática bajo este escenario?

* Por el contrario, sobre los bordes fuertes de la imagen, la varianza local es mucho mayor al ruido ($σ^{2}_{L} ≫ σ^{2}_{\eta}$).
¿Qué valor asume la fracción y cómo protege esto a los bordes de ser difuminados?


##  Ejercicio 5: Eliminación de ruido periódico

Filtraremos interferencias sinusoidales en el dominio frecuencial utilizando filtros Rechaza-banda y Notch.



In [ ]:
# Filtros Rechaza-banda y Notch Frecuenciales

def ruido_periodico(img_path='img_degradada.tif'):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return "Carga 'img_degradada.tif'"

    # 1. Obtener espectro de Fourier
    dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)
    dft_shift = np.fft.fftshift(dft)
    mag = 20 * np.log(cv2.magnitude(dft_shift[:,:,0], dft_shift[:,:,1]) + 1)

    # 2. Localizar picos del ruido (las "estrellas" brillantes fuera del centro)
    # TODO: Encuentra las coordenadas (u, v) de los picos. Puedes hardcodearlos tras inspeccionar visualmente.

    filas, cols = img.shape
    mask_notch = np.ones((filas, cols, 2), np.float32)

    # 3. Crear filtro Notch Ideal
    # TODO: Dibuja círculos negros (ceros) en las coordenadas (u,v) del ruido y en sus conjugados (-u, -v)
    # cv2.circle(mask_notch, (coord_x, coord_y), radio, (0,0), -1)

    # 4. Filtrar y transformar de vuelta
    # fshift_filtrado = dft_shift * mask_notch
    # img_restaurada = ... (idft)

    # 5. Imagen de SOLO RUIDO
    # TODO: Aplica un filtro pasabanda o un notch PASANTE (invirtiendo la máscara anterior)
    # img_solo_ruido = ...

    # Visualizar Espectro, Máscara, Restaurada y Solo Ruido
    pass

**Reflexiona sobre lo que observas:**

* Observe el espectro de la imagen degradada. ¿Por qué el ruido periódico (sinusoidal) no aparece en el origen (centro del espectro) sino como pares de impulsos simétricos?

* ¿Qué desventaja importante tiene aplicar un filtro rechazabanda general (un anillo completo que elimina un rango de frecuencias) frente a un filtro Notch puntual para este tipo específico de degradación?

##  Ejercicio 6: Restauración por filtrado de Wiener (Deconvolución)

Traduciremos la lógica de restauración matemática que combate el desenfoque por movimiento o fuera de foco (de-convolución)

---
* Migrar el tutorial de OpenCV en C++ a Python:
   * [Out of focus deblur](https://docs.opencv.org/trunk/de/d3c/tutorial_out_of_focus_deblur_filter.html)
   * [Motion deblur](https://docs.opencv.org/trunk/d1/dfd/tutorial_motion_deblur_filter.html)


In [ ]:
# Filtro de Wiener]
def filtro_wiener():

    # Pasos generales a implementar en Fourier:
    # 1. Calcular la TDF de la imagen degradada G(u,v)
    # 2. Generar el modelo de degradación H(u,v) (e.g., disco para fuera de foco, línea para movimiento)
    # 3. Calcular la función del filtro de Wiener: W(u,v) = H*(u,v) / (|H(u,v)|^2 + 1/SNR)
    # 4. Estimar F_hat(u,v) = W(u,v) * G(u,v)
    # 5. TDF Inversa para recuperar f_hat(x,y)

    # TODO: Implementación en Python. Testear con imágenes de internet que tengan las degradaciones o genere algunas con IA.
    pass

**Reflexiona sobre lo que observas:**

Un filtro inverso simple intentaría recuperar la imagen haciendo:
$$F(u,v)=G(u,v)/H(u,v)$$

¿Por qué este enfoque básico suele fracasar catastróficamente en la práctica y qué problema resuelve la inclusión del parámetro señal-a-ruido ($SNR$) en el denominador del Filtro de Wiener?


## Ejercicio 7: Trabajo de Aplicación

Selección de estrategias sobre imágenes (FAMILIA_a.jpg, FAMILIA_b.jpg, FAMILIA_c.jpg)

---

* Seleccionar un parche o ROI de la imagen que debería ser homogéneo
* Dibujar histograma de ROI para inferir la distribución estadística
* Obtener parámetros del ruido
* Basado en el tipo de ruido detectado (Gaussiano, Impulsivo, Periódico, etc.), seleccionar el filtro espacial o frecuencial programado en ejercicios anteriores y ajustar los parámetros.

In [ ]:
# Análisis de Imágenes

def aplicacion_familia(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return f"Carga {img_path}"

    # 1. Identificar ruido
    # TODO: Seleccionar un parche o ROI de la imagen que debería ser homogéneo (ej: pared de fondo)
    # roi = img[y1:y2, x1:x2]
    # Dibujar histograma de ROI para inferir la distribución estadística.

    # 2. Elegir y aplicar filtro adecuado
    # TODO: Basado en el tipo de ruido detectado (Gaussiano, Impulsivo, Periódico, etc.),
    # seleccionar el filtro espacial o frecuencial programado en ejercicios anteriores.

    # img_restaurada = ...
    pass

**Reflexiona sobre lo que observas:**

* Al extraer una pequeña región de interés (ROI) que se supone homogénea en la fotografía, ¿a cuál de las distribuciones teóricas estudiadas en el Ejercicio 1 (gaussiana, exponencial, sal/pimienta) se asemeja su histograma?

* En base a su hallazgo, justifique por qué el filtro que acaba de elegir es el matemáticamente más apto para limpiar esa fotografía en particular.

## Extras

Revise temas adicionales a la guía como **Filtro Bilateral** (*BilateralFilter y adaptiveBilateralFilter*) y métodos **Non-Local Means** (*FastNlMeansDenoising y FastNlMeansDenoisingColored*) de OpenCV que son técnicas avanzadas que preservan bordes con mucha fidelidad.



